[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.3_sagemaker/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.3_sagemaker/lab.ipynb)

# 7.3 SageMaker for LLM Serving - Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.3_sagemaker/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud.coreweavex.com/hub/user-redirect/lab/tree/llm-inference-at-scale/content/08_serving/07.3_sagemaker/lab.ipynb)

This lab explores SageMaker endpoint configuration, cost modeling, and autoscaling behavior for LLM workloads.

In [ ]:
# Install dependencies for SageMaker config generation and analysis
import subprocess
subprocess.run(['pip', 'install', '-q', 'matplotlib', 'numpy'], check=True)

# Import libraries for computation and visualization
import numpy as np  # Numerical computation
import matplotlib.pyplot as plt  # Plotting

## Exercise 1: Instance Selection Cost Model

Compare cost-per-token across SageMaker ML instance types for a 7B model.

In [ ]:
def cost_per_million_tokens():
    """Calculate and compare $/M tokens across SageMaker instance types."""
    # Instance catalog: (name, GPU, VRAM_GB, hourly_cost, tokens_per_sec)
    instances = [
        ('ml.g5.xlarge', 'A10G', 24, 1.41, 45),  # Single A10G, budget option
        ('ml.g5.2xlarge', 'A10G', 24, 1.89, 50),  # More CPU/RAM, same GPU
        ('ml.g5.12xlarge', '4xA10G', 96, 7.09, 180),  # Multi-GPU, high throughput
        ('ml.p4d.24xlarge', '8xA100', 320, 37.69, 800),  # Premium, max throughput
        ('ml.g6.xlarge', 'L4', 24, 1.07, 40),  # Newest gen, cost-efficient
    ]

    names = []  # Instance names for chart labels
    costs = []  # Cost per million tokens for each instance
    colors = []  # Bar colors based on cost tier

    for name, gpu, vram, hourly, tps in instances:
        # Calculate cost per million output tokens
        tokens_per_hour = tps * 3600  # Convert tokens/sec to tokens/hour
        cost_per_m = (hourly / tokens_per_hour) * 1_000_000  # $/M tokens
        names.append(f'{name}\n({gpu})')  # Label with GPU type
        costs.append(cost_per_m)  # Store computed cost
        # Color code: green=cheap, amber=mid, rose=expensive
        if cost_per_m < 10:
            colors.append('#dcfce7')  # Green: under $10/M tokens
        elif cost_per_m < 15:
            colors.append('#fef3c7')  # Amber: $10-15/M tokens
        else:
            colors.append('#ffe4e6')  # Rose: over $15/M tokens

    # Create bar chart comparing cost per million tokens
    fig_1, ax_1 = plt.subplots(figsize=(10, 5))
    bars = ax_1.bar(names, costs, color=colors, edgecolor='#000', linewidth=1.2)

    # Add cost labels on top of each bar
    for bar, cost in zip(bars, costs):
        ax_1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'${cost:.2f}', ha='center', fontsize=10)  # Dollar amount label

    ax_1.set_ylabel('Cost per Million Output Tokens ($)')  # Y-axis label
    ax_1.set_title('SageMaker Instance Cost Comparison (7B Model, On-Demand)')  # Title
    ax_1.set_ylim(0, max(costs) * 1.2)  # Add headroom for labels
    plt.tight_layout()  # Clean layout
    plt.show()  # Render chart

# Run the cost comparison
cost_per_million_tokens()

## Exercise 2: Endpoint Configuration Generator

Generate SageMaker endpoint configuration for different deployment patterns.

In [ ]:
def generate_endpoint_config(
    model_name: str,  # Model identifier
    instance_type: str,  # SageMaker ML instance type
    instance_count: int,  # Initial instance count
    container_image: str,  # DLC or custom container URI
    model_s3_uri: str,  # S3 path to model artifacts
    tp_degree: int = 1,  # Tensor parallel degree (GPUs per model copy)
    max_concurrent: int = 64,  # Max concurrent requests per instance
) -> dict:
    """Generate SageMaker endpoint configuration dictionary."""
    # Build serving.properties for DJL container
    serving_properties = {
        'engine': 'Python',  # Use Python engine for vLLM backend
        'option.model_id': model_s3_uri,  # Where to find model weights
        'option.tensor_parallel_degree': tp_degree,  # Multi-GPU parallelism
        'option.max_rolling_batch_size': max_concurrent,  # Continuous batching limit
        'option.rolling_batch': 'vllm',  # Use vLLM as the serving backend
        'option.dtype': 'fp16',  # Half precision for inference
    }

    # Build the endpoint configuration structure
    config = {
        'EndpointConfigName': f'{model_name}-config',  # Config resource name
        'ProductionVariants': [{
            'VariantName': 'primary',  # Default variant name
            'ModelName': model_name,  # Reference to model resource
            'InstanceType': instance_type,  # GPU instance type
            'InitialInstanceCount': instance_count,  # Starting fleet size
            'ContainerStartupHealthCheckTimeoutInSeconds': 600,  # 10 min for model load
            'ModelDataDownloadTimeoutInSeconds': 900,  # 15 min for large models
        }],
        'ServingProperties': serving_properties,  # DJL/vLLM configuration
    }
    return config

# Generate config for Llama-3.1-8B on g5.xlarge
config_8b = generate_endpoint_config(
    model_name='llama-3-8b',  # Model identifier
    instance_type='ml.g5.xlarge',  # Single A10G GPU
    instance_count=2,  # Start with 2 instances for HA
    container_image='763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.27.0-lmi',
    model_s3_uri='s3://my-bucket/llama-3.1-8b/',  # Model location
    tp_degree=1,  # Single GPU, no tensor parallelism
    max_concurrent=64,  # Batch size limit
)

# Print configuration
import json
print(json.dumps(config_8b, indent=2))  # Pretty-print the config

# Generate config for Llama-3.1-70B on p4d.24xlarge (8 GPUs)
config_70b = generate_endpoint_config(
    model_name='llama-3-70b',  # Larger model
    instance_type='ml.p4d.24xlarge',  # 8x A100 GPUs
    instance_count=1,  # Single instance (expensive)
    container_image='763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.27.0-lmi',
    model_s3_uri='s3://my-bucket/llama-3.1-70b/',  # Model location
    tp_degree=8,  # Tensor parallel across all 8 GPUs
    max_concurrent=32,  # Lower batch for larger model
)
print('\n--- 70B Config ---')
print(json.dumps(config_70b, indent=2))  # Print 70B config

## Exercise 3: Autoscaling Policy Comparison

Simulate and compare different SageMaker autoscaling policies for LLM endpoints.

In [ ]:
def simulate_sagemaker_scaling(
    traffic: np.ndarray,  # Requests per second over time
    policy: str,  # 'target_tracking' or 'step_scaling'
    target_value: float,  # Target metric value for scaling decisions
    capacity_per_instance: int,  # Requests each instance can handle
    scale_in_cooldown: int,  # Seconds before scale-in allowed
    scale_out_cooldown: int,  # Seconds before scale-out evaluates
    min_capacity: int,  # Minimum instance count
    max_capacity: int,  # Maximum instance count
) -> tuple:
    """Simulate SageMaker autoscaling response to traffic."""
    n = len(traffic)  # Number of time steps
    instances = np.zeros(n)  # Instance count over time
    latency = np.zeros(n)  # Simulated P99 latency
    instances[0] = min_capacity  # Start at minimum
    last_scale_out = -scale_out_cooldown  # Allow immediate first scale
    last_scale_in = -scale_in_cooldown  # Allow immediate first scale-in

    for t in range(1, n):
        # Calculate utilization ratio
        current_capacity = instances[t-1] * capacity_per_instance
        utilization = traffic[t] / max(current_capacity, 1)  # Avoid division by zero

        # Simulate latency increase under load (exponential above 80%)
        if utilization < 0.8:
            latency[t] = 0.5 + utilization * 0.5  # Linear region: 0.5-0.9s
        else:
            latency[t] = 0.9 + (utilization - 0.8) ** 2 * 50  # Exponential above 80%

        if policy == 'target_tracking':
            # Scale out if utilization exceeds target
            if utilization > target_value and (t - last_scale_out) > scale_out_cooldown:
                needed = int(np.ceil(traffic[t] / (capacity_per_instance * target_value)))
                instances[t] = min(needed, max_capacity)  # Cap at max
                last_scale_out = t  # Reset cooldown
            # Scale in if utilization well below target
            elif utilization < target_value * 0.5 and (t - last_scale_in) > scale_in_cooldown:
                instances[t] = max(min_capacity, instances[t-1] - 1)  # Remove one
                last_scale_in = t  # Reset cooldown
            else:
                instances[t] = instances[t-1]  # No change
        else:  # step_scaling
            # Step function: add 2 instances per threshold breach
            if utilization > 0.9 and (t - last_scale_out) > scale_out_cooldown:
                instances[t] = min(instances[t-1] + 2, max_capacity)
                last_scale_out = t
            elif utilization > 0.7 and (t - last_scale_out) > scale_out_cooldown:
                instances[t] = min(instances[t-1] + 1, max_capacity)
                last_scale_out = t
            elif utilization < 0.3 and (t - last_scale_in) > scale_in_cooldown:
                instances[t] = max(min_capacity, instances[t-1] - 1)
                last_scale_in = t
            else:
                instances[t] = instances[t-1]

    return instances, latency

# Generate traffic pattern: gradual ramp with spike
np.random.seed(42)  # Reproducible
t_steps = 600  # 10 minutes
traffic_sm = np.concatenate([
    np.full(150, 20),  # Low baseline
    np.linspace(20, 100, 100),  # Gradual ramp
    np.full(100, 100),  # Sustained peak
    np.full(50, 150),  # Spike
    np.linspace(150, 20, 100),  # Decay
    np.full(100, 20),  # Return to baseline
])

# Run both policies
inst_tt, lat_tt = simulate_sagemaker_scaling(
    traffic_sm, 'target_tracking', target_value=0.7,
    capacity_per_instance=20, scale_in_cooldown=120,
    scale_out_cooldown=60, min_capacity=1, max_capacity=10)

inst_ss, lat_ss = simulate_sagemaker_scaling(
    traffic_sm, 'step_scaling', target_value=0.7,
    capacity_per_instance=20, scale_in_cooldown=120,
    scale_out_cooldown=60, min_capacity=1, max_capacity=10)

# Plot comparison
fig_3, axes_3 = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

# Traffic
axes_3[0].plot(traffic_sm, color='#2563eb', linewidth=1.5)  # Traffic line
axes_3[0].fill_between(range(len(traffic_sm)), traffic_sm, alpha=0.15, color='#2563eb')
axes_3[0].set_ylabel('Requests/sec')  # Label
axes_3[0].set_title('SageMaker Autoscaling: Target Tracking vs Step Scaling')  # Title

# Instances
axes_3[1].step(range(len(inst_tt)), inst_tt, color='#166534', linewidth=2, label='Target Tracking')
axes_3[1].step(range(len(inst_ss)), inst_ss, color='#991b1b', linewidth=2, label='Step Scaling')
axes_3[1].set_ylabel('Instances')  # Label
axes_3[1].legend()  # Show legend

# Latency
axes_3[2].plot(lat_tt, color='#166534', linewidth=1.2, label='TT Latency')  # Target tracking
axes_3[2].plot(lat_ss, color='#991b1b', linewidth=1.2, label='Step Latency')  # Step scaling
axes_3[2].axhline(y=2.0, color='#64748b', linestyle='--', label='SLO (2s)')  # SLO line
axes_3[2].set_ylabel('P99 Latency (s)')  # Label
axes_3[2].set_xlabel('Time (seconds)')  # X-axis
axes_3[2].legend()  # Show legend

plt.tight_layout()  # Clean layout
plt.show()  # Render